# AKI->CKD Data Exploration

See what data we're working with and where it is. 

Calculate the Alberta Score.

Get XGBoost results.

In [ ]:
import os
import re
import random
import math
import time
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.metrics import precision_recall_curve, average_precision_score

from xgboost import XGBClassifier
from datetime import datetime

from src.alberta_score import *

random.seed(1202)
np.random.seed(1202) 

# which features to include for the final model
hing_features = True
alberta_features = False

Current Working Directory: /data/kidney/Sacha
Parent Directory: /data/kidney


/data/kidney/Sacha/kidneyvenv/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [ ]:
#  -*- load in the features.csv file -*-

# Construct the file path
file_path = "/data/kidney/Hing/features.csv"

# Load the CSV file into a DataFrame
features_df = pd.read_csv(file_path)  # contains all the features that hing engineered

# cast AdmitDt and DischDt to datetime
features_df['admit_date'] = pd.to_datetime(features_df['admit_date'])
features_df['discharge_date'] = pd.to_datetime(features_df['discharge_date'])

# drop patients who died before they were discharged
features_df.drop(features_df[features_df['death_date'] <= features_df['discharge_date']].index, inplace=True)

if hing_features:
    features_df['admit_to_stage1'] = (pd.to_datetime(features_df['stage1_date']) - pd.to_datetime(features_df['admit_date'])).dt.days
    features_df['stage1_to_discharge'] = (pd.to_datetime(features_df['discharge_date']) - pd.to_datetime(features_df['stage1_date'])).dt.days
    features_df['admit_to_stage2'] = (pd.to_datetime(features_df['stage2_date']) - pd.to_datetime(features_df['admit_date'])).dt.days
    features_df['stage2_to_discharge'] = (pd.to_datetime(features_df['discharge_date']) - pd.to_datetime(features_df['stage2_date'])).dt.days
    features_df['admit_to_stage3'] = (pd.to_datetime(features_df['stage3_date']) - pd.to_datetime(features_df['admit_date'])).dt.days
    features_df['stage3_to_discharge'] = (pd.to_datetime(features_df['discharge_date']) - pd.to_datetime(features_df['stage3_date'])).dt.days

# contains only what we need for the Alberta score + ckd_stage45
alberta_df = features_df[["patient_id", "admit_date", "discharge_date", "sex", "age_admit", "highest_stage", "ckd_stage45"]]
alberta_df = alberta_df.rename(columns={"sex": "sex_raw", "age_admit": "age_admit_raw", "highest_stage": "highest_stage_raw"})

alberta_df

,patient_id,admit_date,discharge_date,sex_raw,age_admit_raw,highest_stage_raw,ckd_stage45
0,1,2021-10-19,2021-10-28,1,40,1,0
1,2,2021-03-20,2021-03-25,1,84,1,0
2,3,2020-01-23,2020-02-10,1,83,1,0
3,4,2020-03-11,2020-03-14,0,43,1,0
4,5,2021-01-24,2021-03-02,0,69,2,0
...,...,...,...,...,...,...,...
4689,4690,2021-11-12,2021-11-23,0,51,1,0
4690,4691,2021-05-24,2021-05-27,1,85,1,0
4691,4692,2021-07-06,2021-07-20,0,46,1,0
4692,4693,2021-01-12,2021-01-19,0,84,1,0


In [3]:
features_df

,patient_id,admit_date,discharge_date,sex,age_admit,total_los,stage1,stage1_date,stage1_creatinine,stage2,...,pre-index_medication:nsaids,pre-index_medication:ppi,pre-index_medication:sglt2,pre-index_medication:vancomycin,admit_to_stage1,stage1_to_discharge,admit_to_stage2,stage2_to_discharge,admit_to_stage3,stage3_to_discharge
0,1,2021-10-19,2021-10-28,1,40,9,1.0,2021-10-22,88.0,NaN,...,0.0,0.0,0.0,0.0,3.0,6.0,NaN,NaN,NaN,NaN
1,2,2021-03-20,2021-03-25,1,84,5,1.0,2021-03-20,97.0,NaN,...,0.0,0.0,0.0,0.0,0.0,5.0,NaN,NaN,NaN,NaN
2,3,2020-01-23,2020-02-10,1,83,18,1.0,2020-02-03,85.0,NaN,...,0.0,0.0,0.0,0.0,11.0,7.0,NaN,NaN,NaN,NaN
3,4,2020-03-11,2020-03-14,0,43,3,1.0,2020-03-11,124.0,NaN,...,0.0,1.0,0.0,0.0,0.0,3.0,NaN,NaN,NaN,NaN
4,5,2021-01-24,2021-03-02,0,69,37,1.0,2021-01-25,109.0,1.0,...,0.0,0.0,0.0,0.0,1.0,36.0,22.0,15.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4689,4690,2021-11-12,2021-11-23,0,51,11,1.0,2021-11-13,116.0,NaN,...,NaN,NaN,NaN,NaN,1.0,10.0,NaN,NaN,NaN,NaN
4690,4691,2021-05-24,2021-05-27,1,85,3,1.0,2021-05-24,88.0,NaN,...,0.0,1.0,0.0,0.0,0.0,3.0,NaN,NaN,NaN,NaN
4691,4692,2021-07-06,2021-07-20,0,46,14,1.0,2021-07-16,147.0,NaN,...,0.0,1.0,0.0,0.0,10.0,4.0,NaN,NaN,NaN,NaN
4692,4693,2021-01-12,2021-01-19,0,84,7,1.0,2021-01-14,149.0,NaN,...,0.0,1.0,0.0,0.0,2.0,5.0,NaN,NaN,NaN,NaN


In [4]:
# -*- load in the lab tests -*-

# Load in the index labs
index_labs_file_path = os.path.join(parent_dir, 'Hing', 'in-hosp labs.csv')
index_labs_df = pd.read_csv(index_labs_file_path)

# Load the pre-index labs
prehosp_labs_file_path = os.path.join(parent_dir, 'Hing', 'pre-hosp labs.csv')
prehosp_labs_df = pd.read_csv(prehosp_labs_file_path)

# combine the two
all_labs_df = pd.concat([index_labs_df, prehosp_labs_df], ignore_index=True)

# Cast test_date, AdmitDt, and DischDt to datetime
all_labs_df['test_date'] = pd.to_datetime(all_labs_df['test_date'])
all_labs_df['AdmitDt'] = pd.to_datetime(all_labs_df['AdmitDt'])
all_labs_df['DischDt'] = pd.to_datetime(all_labs_df['DischDt'])

all_labs_df

,test_date,TEST_NM,TEST_RSLT,TEST_UOFM,lab_test_category,AdmitDt,DischDt,id
0,2021-10-19,Creatinine,57,umol/L,Creatinine,2021-10-19,2021-10-28,1
1,2021-10-19,Hemoglobin,123,g/L,Hemoglobin,2021-10-19,2021-10-28,1
2,2021-10-19,eGFR,112,mL/min/1.73m2,eGFR,2021-10-19,2021-10-28,1
3,2021-10-22,Creatinine,88,umol/L,Creatinine,2021-10-19,2021-10-28,1
4,2021-10-22,Hemoglobin,84,g/L,Hemoglobin,2021-10-19,2021-10-28,1
...,...,...,...,...,...,...,...,...
710814,2020-02-28,Creatinine,138,umol/L,Creatinine,2021-01-12,2021-01-19,4693
710815,2020-02-28,eGFR,40,mL/min/1.73m2,eGFR,2021-01-12,2021-01-19,4693
710816,2020-09-30,Creatinine,130,umol/L,Creatinine,2021-01-12,2021-01-19,4693
710817,2020-09-30,Hemoglobin,137,g/L,Hemoglobin,2021-01-12,2021-01-19,4693


In [5]:
# -*- alberta score points -*-

# sex
alberta_df["sex_points"] = alberta_df.sex_raw.map({1: 3, 0: 0})

# age
alberta_df["age_admit_points"] = alberta_df.age_admit_raw.map(age_mapping)

# highest stage
alberta_df["highest_stage_points"] = alberta_df.highest_stage_raw.map(stage_mapping)

# baseline creatinine
alberta_df['baseline_creatinine_raw'] = alberta_df.apply(
    lambda row: get_baseline_creatinine(row['patient_id'], all_labs_df, row['admit_date']), axis=1
)
alberta_df["baseline_creatinine_points"] = alberta_df.baseline_creatinine_raw.map(baseline_creatinine_mapping)

# discharge creatinine
alberta_df['discharge_creatinine_raw'] = alberta_df.apply(
    lambda row: get_discharge_creatinine(row['patient_id'], all_labs_df, row['admit_date'], row['discharge_date']), axis=1
)
alberta_df["discharge_creatinine_points"] = alberta_df.discharge_creatinine_raw.map(discharge_creatinine_mapping)

# albuminuria status
alberta_df['albuminuria_status_raw'] = alberta_df.apply(
    lambda row: get_albuminuria_status(row['patient_id'], all_labs_df, row['admit_date'], row['discharge_date'])[0], axis=1
)
alberta_df["albuminuria_status_points"] = alberta_df.albuminuria_status_raw.map(albuminuria_status_mapping)


In [6]:
alberta_df


,patient_id,admit_date,discharge_date,sex_raw,age_admit_raw,highest_stage_raw,ckd_stage45,sex_points,age_admit_points,highest_stage_points,baseline_creatinine_raw,baseline_creatinine_points,discharge_creatinine_raw,discharge_creatinine_points,albuminuria_status_raw,albuminuria_status_points
0,1,2021-10-19,2021-10-28,1,40,1,0,3,0,0,None,NaN,0.486425,0,unmeasured,1
1,2,2021-03-20,2021-03-25,1,84,1,0,3,2,0,0.871041,2.0,0.882353,0,normal,0
2,3,2020-01-23,2020-02-10,1,83,1,0,3,2,0,0.791855,1.0,0.723982,0,unmeasured,1
3,4,2020-03-11,2020-03-14,0,43,1,0,0,0,0,0.893665,2.0,0.848416,0,unmeasured,1
4,5,2021-01-24,2021-03-02,0,69,2,0,0,2,1,0.882353,2.0,0.757919,0,normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4689,4690,2021-11-12,2021-11-23,0,51,1,0,0,1,0,None,NaN,0.904977,0,mild,1
4690,4691,2021-05-24,2021-05-27,1,85,1,0,3,2,0,0.837104,2.0,0.509050,0,unmeasured,1
4691,4692,2021-07-06,2021-07-20,0,46,1,0,0,0,0,1.221719,4.0,1.640271,7,unmeasured,1
4692,4693,2021-01-12,2021-01-19,0,84,1,0,0,2,0,1.470588,5.0,1.368778,6,unmeasured,1


* We're looking at a couple different tests to calculate albuminuria within `all_labs_df`
* "Albuminuria was characterized by urine albumin:creatinine ratio (ACR) or dipstick using random spot urine measurements during the index admission or for up to 6 months prior to admission. We used urine ACR measurements preferentially, with urine dipstick measurements added for patients without ACR measurements in the derivation cohort. We defined albuminuria categories as normal (ACR, <30 mg/g or urine-dipstick negative), mild (ACR, 30-300 mg/g or urine dipstick trace or 1+), or heavy (ACR, >300 mg/g or urine dipstick positive ≥2+) and characterized patients with multiple tests using the median of multiple measurements, recognizing that albuminuria can be transient". Steps to calculate albuminuria status for a particular patient:
  * Retrieve all the test results that match `Albumin.+Creatinine|ACR`
    * Does "EXT Microalbumin/Creatinine-mg/mmol" count...?
  * The results in the lab dataset for ACR are all in mg/mmol. Convert these to mg/g to use
  * Look 
  * If we don't have the proper measurements for this, we look at uring dipstick measurements. 

In [7]:
# all_labs_df[all_labs_df['TEST_NM'].str.contains('Albumin.+Creatinine|ACR', case=False)].TEST_UOFM.value_counts()

In [8]:
# %pip install matplotlib

In [9]:
# # Filter the DataFrame for "Albumin/Creatinine Ratio" category
# numeric_values = pd.to_numeric(all_labs_df[all_labs_df['lab_test_category'] == "Albumin/Creatinine Ratio"].TEST_RSLT, errors='coerce')

# numeric_values = numeric_values.dropna()

# numeric_values = numeric_values.astype(float)

# import matplotlib.pyplot as plt

# # Plot a histogram of the numeric values
# plt.hist(numeric_values, bins=30, edgecolor='black')
# plt.title('Histogram of Albumin/Creatinine Ratio Values')
# plt.xlabel('Value')
# plt.ylabel('Frequency')
# plt.show()

In [10]:
# # all_labs_df[all_labs_df['TEST_NM'] == "EXT Microalbumin/Creatinine-mg/mmol"]
# all_labs_df[all_labs_df['lab_test_category'] == "Albumin/Creatinine Ratio"].TEST_NM.value_counts()


In [11]:
# # ask about this
# all_labs_df[(all_labs_df.TEST_NM == "Protein Urine UA")&(all_labs_df.TEST_RSLT == "0.30")]

In [12]:
# filtered_rows = all_labs_df[
#     (all_labs_df['lab_test_category'] == "dipstick UA") &
#     (~all_labs_df['TEST_RSLT'].str.contains(r'\+|negative', case=False, na=False))
# ]
# filtered_rows.TEST_UOFM.value_counts()

In [13]:
alberta_df


,patient_id,admit_date,discharge_date,sex_raw,age_admit_raw,highest_stage_raw,ckd_stage45,sex_points,age_admit_points,highest_stage_points,baseline_creatinine_raw,baseline_creatinine_points,discharge_creatinine_raw,discharge_creatinine_points,albuminuria_status_raw,albuminuria_status_points
0,1,2021-10-19,2021-10-28,1,40,1,0,3,0,0,None,NaN,0.486425,0,unmeasured,1
1,2,2021-03-20,2021-03-25,1,84,1,0,3,2,0,0.871041,2.0,0.882353,0,normal,0
2,3,2020-01-23,2020-02-10,1,83,1,0,3,2,0,0.791855,1.0,0.723982,0,unmeasured,1
3,4,2020-03-11,2020-03-14,0,43,1,0,0,0,0,0.893665,2.0,0.848416,0,unmeasured,1
4,5,2021-01-24,2021-03-02,0,69,2,0,0,2,1,0.882353,2.0,0.757919,0,normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4689,4690,2021-11-12,2021-11-23,0,51,1,0,0,1,0,None,NaN,0.904977,0,mild,1
4690,4691,2021-05-24,2021-05-27,1,85,1,0,3,2,0,0.837104,2.0,0.509050,0,unmeasured,1
4691,4692,2021-07-06,2021-07-20,0,46,1,0,0,0,0,1.221719,4.0,1.640271,7,unmeasured,1
4692,4693,2021-01-12,2021-01-19,0,84,1,0,0,2,0,1.470588,5.0,1.368778,6,unmeasured,1


In [14]:
# -*- drop patients who don't have a baseline creatinine and discharge creatinine

alberta_df = alberta_df.dropna(subset=['baseline_creatinine_raw'])
alberta_df = alberta_df.dropna(subset=['discharge_creatinine_raw'])

In [15]:
# -*- calculate alberta score 

alberta_df["alberta_score"] = alberta_df["sex_points"]+\
                              alberta_df["age_admit_points"]+\
                              alberta_df["highest_stage_points"]+\
                              alberta_df["baseline_creatinine_points"]+\
                              alberta_df["discharge_creatinine_points"]+\
                              alberta_df["albuminuria_status_points"]

In [16]:
len(alberta_df.patient_id.unique().tolist())

3820

In [17]:
# -*- standardize feature set patient list and order -*-

# Filter features_df to include only patients present in alberta_df
features_df = features_df[features_df['patient_id'].isin(alberta_df['patient_id'])]

# Filter alberta_df to include only patients present in features_df
alberta_df = alberta_df[alberta_df['patient_id'].isin(features_df['patient_id'])]

# Sort both DataFrames by patient_id
features_df = features_df.sort_values(by='patient_id').reset_index(drop=True)
alberta_df = alberta_df.sort_values(by='patient_id').reset_index(drop=True)

# Ensure they have the same shape
assert features_df.shape[0] == alberta_df.shape[0], "DataFrames do not have the same number of rows."

In [18]:
# -*- create features_used, feature_names, attributes_used -*-
# Hing used "attribute" to refer to the target variable

alberta_points_features = alberta_df[[
    "sex_points",
    "age_admit_points",
    "highest_stage_points",
    "baseline_creatinine_points",
    "discharge_creatinine_points",
    "albuminuria_status_points"
]]

if hing_features:  # use hing's features
    if alberta_features:
        # add the alberta score features to the feature set
        features_df = pd.concat([features_df, alberta_points_features], axis=1)

    # Hing code
    title_list = list(features_df.dtypes.index)
    dataframe_values = features_df.values
    attribute_index = list(features_df.dtypes.index).index('ckd_stage45') # only 286 cases are true 
    attributes_used = dataframe_values[:, attribute_index]
    date_indexes = [i for i in range(len(title_list)) if ('_date' in title_list[i])]
    id_indexes = [i for i in range(len(title_list)) if ('patient_id' in title_list[i])]
    feature_columns = [i for i in range(np.shape(dataframe_values)[1]) if not (i in date_indexes+id_indexes) and i != attribute_index]
    features_used = dataframe_values[:, feature_columns] # (4694, 467)
    feature_names = features_df.dtypes.index[feature_columns]

else:  # we only use alberta score features
    features_used = alberta_points_features.values
    feature_names = alberta_points_features.columns.values
    attributes_used = alberta_df["ckd_stage45"].values


In [19]:
# import matplotlib.pyplot as plt

# # Plot a histogram of the alberta_score column
# plt.hist(alberta_df['alberta_score'].dropna(), bins=20, edgecolor='black')
# plt.title('Frequencies of Alberta Scores')
# plt.xlabel('Alberta Score')
# plt.ylabel('Frequency')
# plt.show()

In [ ]:
# -*- run training loop and get results -*-

# Define the date and feature type for the subfolder name
current_date = datetime.now().strftime("%Y%m%d")
if alberta_features: feature_type = "alberta"
elif hing_features: feature_type = "hing"
else: feature_type = "all"
extra = ""

subfolder_name = f"{current_date}_{feature_type}_{extra}_fold_results"

# Create the experiments folder and subfolder for this run
os.makedirs(f"experiments/{subfolder_name}", exist_ok=True)

# Initialize variables for storing results across folds
tprs = []  # True positive rates for ROC curve
aucs1 = []  # AUC values for ROC curve
aucs2 = []  # AUC values for Precision-Recall curve
precisions = []  # Precision values for PRC curve
mean_fpr1 = np.linspace(0, 1, 100)  # Mean false positive rates for ROC
mean_fpr2 = np.linspace(0, 1, 100)  # Mean recall values for PRC
i = 0  # Fold counter
accuracy_sum = 0  # Sum of accuracies across folds
sensitivity_sum = 0  # Sum of sensitivities across folds
specificity_sum = 0  # Sum of specificities across folds
ppv_sum = 0  # Sum of positive predictive values across folds
npv_sum = 0  # Sum of negative predictive values across folds
f1_sum = 0  # Sum of F1 scores across folds
y_real = []  # True labels for all folds
y_proba = []  # Predicted probabilities for all folds
# plt.clf()  # Clear any existing plots
best_threshold_roc = 0  # Sum of best thresholds for ROC curve
best_threshold_prc = 0  # Sum of best thresholds for PRC curve

# Initialize 10-fold cross-validation
cv = KFold(n_splits=10, shuffle=True, random_state=1202)
skip = 0

# Perform cross-validation
for i, (train, test) in enumerate(cv.split(features_used, attributes_used)):
	if i < skip:  # Skip the first skip folds (for if training gets halted)
		continue
	
	t0 = time.time()

	print(f"Training fold {i + 1}...")

	X_train, y_train = features_used[train].astype(float), attributes_used[train].astype('int8')
	X_test, y_test = features_used[test].astype(float), attributes_used[test].astype('int8')

	attribute0_count = np.sum(y_train == 0)  # Count of attribute 0 in training set
	attribute1_count = np.sum(y_train == 1)  # Count of attribute 1 in training set

	classifier = XGBClassifier(random_state=1202)  # Initialize the classifier with a random seed
	# classifier = XGBClassifier(random_state=1202, scale_pos_weight = math.sqrt(attribute0_count/attribute1_count), learning_rate = 0.1)  # Initialize the classifier

	feature_names_fold = feature_names

	if hing_features:  # perform RFE on the training set
		rfe = RFE(estimator=classifier, n_features_to_select=240, step=0.1, verbose=1)  # 381 seconds
		rfe = rfe.fit(X_train, y_train) 

		X_train = rfe.transform(X_train)
		X_test = rfe.transform(X_test)

		feature_names_fold = feature_names[rfe.support_]

	# Train the classifier on the training set
	classifier.fit(X_train.astype(float), y_train)

	# Predict on the test set
	y_pred = classifier.predict(X_test)
	y_pred = y_pred.astype('int8')

	# Compute confusion matrix and extract metrics
	cm = confusion_matrix(y_test, y_pred)
	TN = cm[0, 0]  # True negatives
	FP = cm[0, 1]  # False positives
	FN = cm[1, 0]  # False negatives
	TP = cm[1, 1]  # True positives
	P = TP + FN  # Total positives
	N = TN + FP  # Total negatives

	# Calculate performance metrics
	accuracy = (TP + TN) / (P + N)
	sensitivity = TP / P  # Recall
	specificity = TN / N
	ppv = TP / (TP + FP)  # Precision
	npv = TN / (TN + FN)
	f1 = f1_score(y_test, y_pred)

	# Accumulate metrics for averaging later
	accuracy_sum += accuracy
	sensitivity_sum += sensitivity
	specificity_sum += specificity
	ppv_sum += ppv
	npv_sum += npv
	f1_sum += f1

	# Predict probabilities for ROC and PRC analysis
	probas_ = classifier.predict_proba(X_test)

	# Compute ROC curve and find the best threshold
	fpr, tpr, roc_thresholds = roc_curve(y_test, probas_[:, 1])
	gmeans = tpr * (1 - fpr)  # Geometric mean of sensitivity and specificity
	ix1 = np.argmax(gmeans)  # Index of the best threshold
	best_threshold_roc += roc_thresholds[ix1]

	# Compute Precision-Recall curve and find the best threshold
	precision, recall, prc_thresholds = precision_recall_curve(y_test, probas_[:, 1])
	fscore = (2 * precision * recall) / (precision + recall)  # F1 score for PRC
	ix2 = np.argmax(fscore)  # Index of the best threshold
	best_threshold_prc += prc_thresholds[ix2]

	# Interpolate ROC and PRC curves for averaging
	tprs.append(np.interp(mean_fpr1, fpr, tpr))
	precisions.append(np.interp(mean_fpr2, recall, precision))
	y_real.append(y_test)
	y_proba.append(probas_[:, 1])
	tprs[-1][0] = 0.0  # Ensure the first TPR value is 0
	precisions[-1][0] = 1.0  # Ensure the first precision value is 1

	# Calculate AUC for ROC and PRC
	roc_auc = auc(fpr, tpr)
	prc_auc = auc(recall, precision)
	aucs1.append(roc_auc)
	aucs2.append(prc_auc)

	# Save fold-specific results to a JSON file
	fold_results = {
		"accuracy": accuracy,
		"sensitivity": sensitivity,
		"specificity": specificity,
		"ppv": ppv,
		"npv": npv,
		"f1": f1,
		"roc_auc": roc_auc,
		"prc_auc": prc_auc,
		"best_threshold_roc": roc_thresholds[ix1],
		"best_threshold_prc": prc_thresholds[ix2]
	}

	# Convert fold_results values to Python native types for JSON serialization
	fold_results = {key: (value.item() if isinstance(value, np.generic) else value) for key, value in fold_results.items()}
	with open(f"experiments/{subfolder_name}/fold_{i + 1}.json", "w") as f:
		json.dump(fold_results, f, indent=4)

	# save feature importances to a file
	importances = classifier.feature_importances_
	indices = np.argsort(importances)[::-1]
	with open(f"experiments/{subfolder_name}/fold_{i + 1}_feature_importances.txt", 'w') as f:
		for j in range(X_train.shape[1]):
			f.write('%d\t%d\t%s\t%.6f\n' % (j, indices[j], feature_names_fold[indices[j]], importances[indices[j]]))

	print("Fold {} completed in {:.2f} seconds.".format(i + 1, time.time() - t0))
	i += 1  # Increment fold counter

# Print aggregated results over all folds
print("Aggregated Results Over All Folds:")
print(f"Mean Accuracy: {accuracy_sum / 10:.4f}")
print(f"Mean Sensitivity: {sensitivity_sum / 10:.4f}")
print(f"Mean Specificity: {specificity_sum / 10:.4f}")
print(f"Mean PPV: {ppv_sum / 10:.4f}")
print(f"Mean NPV: {npv_sum / 10:.4f}")
print(f"Mean F1 Score: {f1_sum / 10:.4f}")
print(f"Mean ROC Best Threshold: {best_threshold_roc / 10:.4f}")
print(f"Mean PRC Best Threshold: {best_threshold_prc / 10:.4f}")
print(f"Mean ROC AUC: {np.mean(aucs1):.4f}")
print(f"Mean PRC AUC: {np.mean(aucs2):.4f}")


Training fold 4...
Fitting estimator with 479 features.
Fitting estimator with 432 features.
Fitting estimator with 385 features.
Fitting estimator with 338 features.
Fitting estimator with 291 features.
Fitting estimator with 244 features.
Fold 4 completed in 1019.18 seconds.
Training fold 5...
Fitting estimator with 479 features.
Fitting estimator with 432 features.
Fitting estimator with 385 features.
Fitting estimator with 338 features.
Fitting estimator with 291 features.
Fitting estimator with 244 features.
Fold 5 completed in 1116.49 seconds.
Training fold 6...
Fitting estimator with 479 features.
Fitting estimator with 432 features.
Fitting estimator with 385 features.
Fitting estimator with 338 features.
Fitting estimator with 291 features.
Fitting estimator with 244 features.


In [ ]:
# #   WORKSHOP CELL

# # filter all_labs_df to only include 

# albuminuria_labs = all_labs_df[(all_labs_df['lab_test_category'].str.contains("Albumin/Creatinine Ratio", case=False))
#                                |
#                                (all_labs_df['lab_test_category'].str.contains('dipstick UA', case=False))]


# albuminuria_labs

# # Perform a left join of albuminuria_labs to alberta_df
# merged_df = alberta_df.merge(albuminuria_labs, how='left', left_on='patient_id', right_on='id')

# filtered_merged_df = merged_df[
#     (merged_df['test_date'].isnull()) |
#     (
#         ((merged_df['test_date'] >= merged_df['admit_date'] - pd.Timedelta(days=180)) & 
#          (merged_df['test_date'] <= merged_df['discharge_date']))
#     )
# ]

# filtered_merged_df




# # # Filter for ACR or albumin tests within the time window
# # acr_pattern = 
# # dipstick_pattern = 

# # acr_labs = patient_labs[
# #     patient_labs['TEST_NM'].str.contains(acr_pattern, case=False) &
# #     (patient_labs['test_date'] >= lower_bound) &
# #     (patient_labs['test_date'] <= index_discharge_date)
# # ]

# # dipstick_labs = patient_labs[
# #     patient_labs['lab_test_category'].str.contains(dipstick_pattern, case=False) &
# #     (patient_labs['test_date'] >= lower_bound) &
# #     (patient_labs['test_date'] <= index_discharge_date)
# # ]

# unique_patient_ids = filtered_merged_df['patient_id'].nunique()

# # Retain only one row per patient_id by keeping the first occurrence
# filtered_merged_df = filtered_merged_df.drop_duplicates(subset='patient_id', keep='first')

# filtered_merged_df

In [ ]:
# # Calculate the proportion of rows with a non-null value in TEST_NM
# proportion_with_test_nm = filtered_merged_df['TEST_NM'].notnull().mean()
# 1-proportion_with_test_nm

In [ ]:
# # this seems like there are too many unmeasured values.

# # Calculate the value counts
# value_counts = alberta_df.albuminuria_status_raw.value_counts()

# # Calculate the proportions
# proportions = value_counts / value_counts.sum()

# # Display the proportions
# proportions

In [ ]:
# test_patient_id = 200
# result, acl, dipstick = get_albuminuria_status(test_patient_id, 
#                        all_labs_df, 
#                        alberta_df.loc[alberta_df['patient_id'] == test_patient_id, 'admit_date'].values[0],
#                        alberta_df.loc[alberta_df['patient_id'] == test_patient_id, 'discharge_date'].values[0])

In [ ]:
# # Count the occurrences of each type in the baseline_creatinine column
# float_count = alberta_df['baseline_creatinine'].apply(lambda x: isinstance(x, float)).sum()
# none_count = alberta_df['baseline_creatinine'].isna().sum()
# invalid_unit_count = (alberta_df['baseline_creatinine'] == "Invalid unit").sum()
# invalid_test_result_count = (alberta_df['baseline_creatinine'] == "Invalid test result").sum()

# print(f"Floats: {float_count}")
# print(f"Nones: {none_count}")
# print(f"Invalid unit: {invalid_unit_count}")
# print(f"Invalid test result: {invalid_test_result_count}")

In [ ]:
# alberta_df

In [ ]:
# get_baseline_creatinine(patient_id, all_labs_df, index_admit_date)

In [ ]:
# print("\n".join(features_df.columns))

patient_id
admit_date
discharge_date
sex
age_admit
total_los
stage1
stage1_date
stage1_creatinine
stage2
stage2_date
stage2_creatinine
stage3
stage3_date
stage3_creatinine
highest_stage
death_date
ckd_stage45
stroke_after
chf_after
mi_after
index_vars:icu
index_vars:goal_acute
index_vars:goal_community
index_vars:goal_transition
index_vars:goal_others
index_vars:goal_intensive
index_vars:goal_mobility
index_vars:goal_assessment
index_vars:goal_perioperative
index_vars:cardiac_surgery
index_vars:insulin
index_vars:beta_blocker
index_vars:covid_test_result
index_vars:smoke
index_vars:cardiac_catheterization
index_vars:ami
index_vars:chf
index_vars:dialysis
index_vars:mechanical_ventilation
index_vars:renal_ultralsound
index_vars:angiogram
index_vars:foley_catheter
index_vars:ot_assessment
index_vars:pt_assessment
index_vars:obstructive_uropathy
index_vars:sepsis
index_vars:general_internal_medicine
index_vars:emergency_medicine
index_vars:nephrology
index_vars:neurology
index_vars:transp

In [ ]:
# features_df['labs_mean:Albumin'].head()

0          NaN
1          NaN
2    35.000000
3          NaN
4    28.666667
Name: labs_mean:Albumin, dtype: float64

In [ ]:
# ############################ get the most recent index creatinine test results
# # assumption: the most recent test result is an appropriate stand-in for "discharge creatinine"

# # Construct the file path for the new CSV file
# index_labs_file_path = os.path.join(parent_dir, 'Hing', 'in-hosp labs.csv')

# # Load the CSV file into a new DataFrame
# index_labs_df = pd.read_csv(index_labs_file_path)

# # Filter the dataframe for TEST_NM = 'Creatinine' and valid test_date <= DischDt
# index_creatine_df = index_labs_df[(index_labs_df['TEST_NM'] == 'Creatinine') & (index_labs_df['test_date'] <= index_labs_df['DischDt'])]

# # Convert test_date to datetime for sorting
# index_creatine_df['test_date'] = pd.to_datetime(index_creatine_df['test_date'])

# # Sort by id and test_date in descending order
# index_creatine_df = index_creatine_df.sort_values(by=['id', 'test_date'], ascending=[True, False])

# # Drop duplicates to keep the most recent record for each id
# index_creatine_df = index_creatine_df.drop_duplicates(subset='id', keep='first')

# index_creatine_df

/tmp/ipykernel_2530344/3644650818.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  index_creatine_df['test_date'] = pd.to_datetime(index_creatine_df['test_date'])


,test_date,TEST_NM,TEST_RSLT,TEST_UOFM,lab_test_category,AdmitDt,DischDt,id
9,2021-10-24,Creatinine,43,umol/L,Creatinine,2021-10-19,2021-10-28,1
28,2021-03-25,Creatinine,78,umol/L,Creatinine,2021-03-20,2021-03-25,2
110,2020-02-10,Creatinine,64,umol/L,Creatinine,2020-01-23,2020-02-10,3
127,2020-03-13,Creatinine,75,umol/L,Creatinine,2020-03-11,2020-03-14,4
319,2021-03-01,Creatinine,67,umol/L,Creatinine,2021-01-24,2021-03-02,5
...,...,...,...,...,...,...,...,...
519594,2021-11-23,Creatinine,80,umol/L,Creatinine,2021-11-12,2021-11-23,4690
519619,2021-05-27,Creatinine,45,umol/L,Creatinine,2021-05-24,2021-05-27,4691
519702,2021-07-19,Creatinine,145,umol/L,Creatinine,2021-07-06,2021-07-20,4692
519753,2021-01-19,Creatinine,121,umol/L,Creatinine,2021-01-12,2021-01-19,4693


In [ ]:
# ############################ get the most recent pre-hospital creatinine test results

# # Construct the file path for the new CSV file
# prehosp_labs_file_path = os.path.join(parent_dir, 'Hing', 'pre-hosp labs.csv')

# # Load the CSV file into a new DataFrame
# prehosp_labs_df = pd.read_csv(prehosp_labs_file_path)

# # Convert test_date to datetime for sorting
# prehosp_labs_df['test_date'] = pd.to_datetime(prehosp_labs_df['test_date'])

# # Filter the dataframe for TEST_NM = 'Creatinine' and valid test_date <= DischDt
# prehosp_creatinine_df = prehosp_labs_df[(prehosp_labs_df['TEST_NM'] == 'Creatinine')]

# # Sort by id and test_date in descending order
# prehosp_creatinine_df = prehosp_creatinine_df.sort_values(by=['id', 'test_date'], ascending=[True, False])

# # Drop duplicates to keep the most recent record for each id
# prehosp_creatinine_df = prehosp_creatinine_df.drop_duplicates(subset='id', keep='first')

# prehosp_creatinine_df

,test_date,TEST_NM,TEST_RSLT,TEST_UOFM,lab_test_category,AdmitDt,DischDt,id
9,2021-03-18,Creatinine,70,umol/L,Creatinine,2021-03-20,2021-03-25,2
20,2019-10-10,Creatinine,70,umol/L,Creatinine,2020-01-23,2020-02-10,3
54,2020-03-10,Creatinine,105,umol/L,Creatinine,2020-03-11,2020-03-14,4
77,2021-01-23,Creatinine,65,umol/L,Creatinine,2021-01-24,2021-03-02,5
90,2021-09-08,Creatinine,102,umol/L,Creatinine,2021-09-09,2021-10-19,7
...,...,...,...,...,...,...,...,...
190819,2021-03-31,Creatinine,105,umol/L,Creatinine,2021-10-26,2021-12-17,4686
190831,2020-03-12,Creatinine,107,umol/L,Creatinine,2020-04-20,2020-04-25,4688
190985,2021-05-23,Creatinine,61,umol/L,Creatinine,2021-05-24,2021-05-27,4691
191024,2021-07-02,Creatinine,124,umol/L,Creatinine,2021-07-06,2021-07-20,4692


In [ ]:
# # Filter the dataframe for TEST_NM = 'Creatinine' and valid test_date <= DischDt
# filtered_df = labs_df[(labs_df['TEST_NM'] == 'Creatinine') & (labs_df['test_date'] <= labs_df['DischDt'])]

# # Convert test_date to datetime for sorting
# filtered_df['test_date'] = pd.to_datetime(filtered_df['test_date'])

# # Sort by id and test_date in descending order
# filtered_df = filtered_df.sort_values(by=['id', 'test_date'], ascending=[True, False])

# # Drop duplicates to keep the most recent record for each id
# most_recent_records = filtered_df.drop_duplicates(subset='id', keep='first')

# most_recent_records

In [ ]:
# # calculate albuminuria

# # combine index_labs_df and prehosp_labs_df
# all_labs_df = pd.concat([index_labs_df, prehosp_labs_df], ignore_index=True)

# all_labs_df

,test_date,TEST_NM,TEST_RSLT,TEST_UOFM,lab_test_category,AdmitDt,DischDt,id
0,2021-10-19,Creatinine,57,umol/L,Creatinine,2021-10-19,2021-10-28,1
1,2021-10-19,Hemoglobin,123,g/L,Hemoglobin,2021-10-19,2021-10-28,1
2,2021-10-19,eGFR,112,mL/min/1.73m2,eGFR,2021-10-19,2021-10-28,1
3,2021-10-22,Creatinine,88,umol/L,Creatinine,2021-10-19,2021-10-28,1
4,2021-10-22,Hemoglobin,84,g/L,Hemoglobin,2021-10-19,2021-10-28,1
...,...,...,...,...,...,...,...,...
710814,2020-02-28 00:00:00,Creatinine,138,umol/L,Creatinine,2021-01-12,2021-01-19,4693
710815,2020-02-28 00:00:00,eGFR,40,mL/min/1.73m2,eGFR,2021-01-12,2021-01-19,4693
710816,2020-09-30 00:00:00,Creatinine,130,umol/L,Creatinine,2021-01-12,2021-01-19,4693
710817,2020-09-30 00:00:00,Hemoglobin,137,g/L,Hemoglobin,2021-01-12,2021-01-19,4693


In [ ]:
# unique_test_nm = all_labs_df['TEST_NM'].unique()
# print(unique_test_nm)

['Creatinine' 'Hemoglobin' 'eGFR' 'Albumin' 'HCO3'
 'C-Reactive Protein (CRP)' 'Glucose' 'Glucose Meter' 'HCO3,VENOUS'
 'GFR ESTIMATED' 'Glucose, Random' 'GLUCOSE RANDOM' 'Phosphate'
 'Cholesterol, Total' 'Urate' 'Bicarbonate, Bld'
 'Glomerular Filtration Rate Estimate' 'Glucose, Bld' 'Glucose, random'
 'Phosphorus' 'Protein Urine UA' 'Protein / Creatinine Ratio, Urine'
 'Albumin / Creatinine Ratio' 'Creatinine Serum' 'C-Reactive Protein'
 'Glucose (mmol/L)' 'Bicarbonate, Venous' 'Cholesterol'
 'Bicarbonate, Arterial' 'C Reactive Protein Quantitative'
 'C Reactive Protein High Sensitivity' 'HCO3, Calculated, Arterial'
 'Hemoglobin, Total' 'Glucose, Blood Gas' 'HCO3,ARTERIAL'
 'Glucose, Patient Glucose Meter' 'Hemoglobin, Arterial' 'Hb Pre'
 'Bicarbonate, Mixed Venous' 'Hemoglobin, Venous' 'Uric Acid'
 'EXT Creatinine' 'EXT Hemoglobin (Hgb)-g/L' 'EXT Glucose-Random-mmol/L'
 'EXT Glucose, Ur' 'HCO3 Calculated' 'HCO3 Calc Arterial' 'Total HGB'
 'Glucose Random' 'EXT HC03-(calc.) - blood g

In [ ]:
# import pandas as pd
# import numpy as np

# def get_baseline_creatinine(patient_id, all_labs_df, index_admit_date):
#     """
#     Get the baseline creatinine value for a patient.
    
#     This function finds the most recent outpatient creatinine measurement 
#     between 7 and 365 days prior to the index hospitalization.
    
#     Parameters:
#     -----------
#     patient_id : int or str
#         The unique identifier for the patient
#     all_labs_df : pandas.DataFrame
#         DataFrame containing lab test results
#     index_admit_date : datetime
#         The admission date for the index hospitalization
    
#     Returns:
#     --------
#     float
#         The baseline creatinine value in mg/dL
#     """
#     # Filter labs for the specific patient
#     patient_labs = all_labs_df[all_labs_df['patient_id'] == patient_id]
    
#     # Filter for creatinine tests only
#     creatinine_labs = patient_labs[patient_labs['TEST_NM'].str.contains('Creatinine', case=False) & 
#                                    ~patient_labs['TEST_NM'].str.contains('Ratio|Protein', case=False)]
    
#     # Calculate the time window (7-365 days before admission)
#     lower_bound = index_admit_date - pd.Timedelta(days=365)
#     upper_bound = index_admit_date - pd.Timedelta(days=7)
    
#     # Filter for tests within the time window
#     window_labs = creatinine_labs[(creatinine_labs['test_date'] >= lower_bound) & 
#                                   (creatinine_labs['test_date'] <= upper_bound)]
    
#     # If no labs in window, return None
#     if window_labs.empty:
#         return None
    
#     # Get the most recent test before admission
#     most_recent = window_labs.sort_values('test_date', ascending=False).iloc[0]
    
#     # Return the creatinine value
#     return most_recent['TEST_RSLT']

# def get_discharge_creatinine(patient_id, all_labs_df, index_discharge_date):
#     """
#     Get the discharge creatinine value for a patient.
    
#     This function finds the last inpatient creatinine measurement before 
#     hospital discharge.
    
#     Parameters:
#     -----------
#     patient_id : int or str
#         The unique identifier for the patient
#     all_labs_df : pandas.DataFrame
#         DataFrame containing lab test results
#     index_discharge_date : datetime
#         The discharge date for the index hospitalization
    
#     Returns:
#     --------
#     float
#         The discharge creatinine value in mg/dL
#     """
#     # Filter labs for the specific patient
#     patient_labs = all_labs_df[all_labs_df['patient_id'] == patient_id]
    
#     # Filter for creatinine tests only
#     creatinine_labs = patient_labs[patient_labs['TEST_NM'].str.contains('Creatinine', case=False) & 
#                                    ~patient_labs['TEST_NM'].str.contains('Ratio|Protein', case=False)]
    
#     # Filter for tests before discharge
#     discharge_labs = creatinine_labs[creatinine_labs['test_date'] <= index_discharge_date]
    
#     # If no labs before discharge, return None
#     if discharge_labs.empty:
#         return None
    
#     # Get the most recent test before discharge
#     most_recent = discharge_labs.sort_values('test_date', ascending=False).iloc[0]
    
#     # Return the creatinine value
#     return most_recent['TEST_RSLT']

# def get_albuminuria_status(patient_id, all_labs_df, index_admit_date):
#     """
#     Determine the albuminuria status for a patient.
    
#     This function checks urine albumin:creatinine ratio (ACR) or urine dipstick 
#     measurements during or 6 months prior to index admission.
    
#     Categories:
#     - Normal: ACR < 30 mg/g or dipstick negative
#     - Mild: ACR 30-300 mg/g or dipstick trace or 1+
#     - Heavy: ACR > 300 mg/g or dipstick positive ≥2+
    
#     Parameters:
#     -----------
#     patient_id : int or str
#         The unique identifier for the patient
#     all_labs_df : pandas.DataFrame
#         DataFrame containing lab test results
#     index_admit_date : datetime
#         The admission date for the index hospitalization
    
#     Returns:
#     --------
#     str
#         The albuminuria category: 'normal', 'mild', 'heavy', or 'unmeasured'
#     """
#     # Filter labs for the specific patient
#     patient_labs = all_labs_df[all_labs_df['patient_id'] == patient_id]
    
#     # Calculate the time window (6 months before admission to admission date)
#     lower_bound = index_admit_date - pd.Timedelta(days=180)
    
#     # Filter for ACR or albumin tests within the time window
#     acr_pattern = 'Albumin.+Creatinine|ACR'
#     dipstick_pattern = 'Protein.+Creatinine|Protein Urine'
    
#     acr_labs = patient_labs[patient_labs['TEST_NM'].str.contains(acr_pattern, case=False) & 
#                           (patient_labs['test_date'] >= lower_bound) & 
#                           (patient_labs['test_date'] <= index_admit_date)]
    
#     dipstick_labs = patient_labs[patient_labs['TEST_NM'].str.contains(dipstick_pattern, case=False) & 
#                                (patient_labs['test_date'] >= lower_bound) & 
#                                (patient_labs['test_date'] <= index_admit_date)]
    
#     # If no labs in window, return 'unmeasured'
#     if acr_labs.empty and dipstick_labs.empty:
#         return 'unmeasured'
    
#     # Prioritize ACR measurements over dipstick
#     if not acr_labs.empty:
#         # Get the median of multiple measurements
#         acr_values = acr_labs['TEST_RSLT'].median()
        
#         if acr_values < 30:  # < 30 mg/g
#             return 'normal'
#         elif acr_values <= 300:  # 30-300 mg/g
#             return 'mild'
#         else:  # > 300 mg/g
#             return 'heavy'
    
#     # Use dipstick if ACR not available
#     if not dipstick_labs.empty:
#         # This is simplified and would need to be adjusted based on how dipstick results are recorded
#         # Assuming numeric values or text values like 'negative', 'trace', '1+', '2+', etc.
#         dipstick_value = dipstick_labs.sort_values('test_date', ascending=False).iloc[0]['TEST_RSLT']
        
#         if dipstick_value == 'negative' or dipstick_value == '0':
#             return 'normal'
#         elif dipstick_value == 'trace' or dipstick_value == '1+':
#             return 'mild'
#         else:  # 2+ or higher
#             return 'heavy'
    
#     return 'unmeasured'

# def calculate_alberta_score(row, all_labs_df):
#     """
#     Calculate the ALBERTA risk score for a patient.
    
#     Based on the 6-variable model from James et al. paper, this function 
#     calculates the risk score with points assigned to:
#     - Age
#     - Sex
#     - Baseline serum creatinine
#     - Albuminuria
#     - Acute kidney injury stage
#     - Discharge serum creatinine
    
#     Parameters:
#     -----------
#     row : pandas.Series
#         A row from the features DataFrame containing patient info
#     all_labs_df : pandas.DataFrame
#         DataFrame containing lab test results
    
#     Returns:
#     --------
#     int
#         The total ALBERTA risk score
#     """
#     score = 0
    
#     # 1. Age points
#     age = row['age_admit']
#     if age < 50:
#         score += 0
#     elif age < 60:
#         score += 1
#     elif age < 70:
#         score += 2
#     elif age < 80:
#         score += 2
#     elif age < 90:
#         score += 2
#     else:  # ≥ 90
#         score += 3
    
#     # 2. Sex points
#     if row['sex'] == 1:  # Male (assuming 1=Male, 0=Female based on the images)
#         score += 0
#     else:  # Female
#         score += 3
    
#     # 3. Baseline serum creatinine points
#     baseline_cr = get_baseline_creatinine(row['patient_id'], all_labs_df, row['AdmitDt'])
#     if baseline_cr is not None:
#         if baseline_cr < 0.6:
#             score += 0
#         elif baseline_cr < 0.7:
#             score += 1
#         elif baseline_cr < 0.8:
#             score += 1
#         elif baseline_cr < 0.9:
#             score += 2
#         elif baseline_cr < 1.0:
#             score += 2
#         elif baseline_cr < 1.1:
#             score += 3
#         elif baseline_cr < 1.2:
#             score += 3
#         elif baseline_cr < 1.3:
#             score += 4
#         else:  # ≥ 1.3
#             score += 5
    
#     # 4. Albuminuria points
#     albuminuria = get_albuminuria_status(row['patient_id'], all_labs_df, row['AdmitDt'])
#     if albuminuria == 'normal':
#         score += 0
#     elif albuminuria == 'mild':
#         score += 1
#     elif albuminuria == 'heavy':
#         score += 3
#     else:  # unmeasured
#         score += 1
    
#     # 5. Acute kidney injury stage points
#     aki_stage = row['highest_stage']  # Assuming this is the KDIGO AKI stage (1, 2, or 3)
#     if aki_stage == 1:
#         score += 0
#     elif aki_stage == 2:
#         score += 1
#     else:  # Stage 3
#         score += 3
    
#     # 6. Discharge serum creatinine points
#     discharge_cr = get_discharge_creatinine(row['patient_id'], all_labs_df, row['DischDt'])
#     if discharge_cr is not None:
#         if discharge_cr < 1.0:
#             score += 0
#         elif discharge_cr < 1.3:
#             score += 3
#         elif discharge_cr < 1.6:
#             score += 6
#         elif discharge_cr < 1.9:
#             score += 7
#         else:  # ≥ 1.9
#             score += 11
    
#     return score

# def predict_ckd_risk(score):
#     """
#     Convert ALBERTA score to predicted risk of advanced CKD.
    
#     Based on the risk categories from the paper:
#     - Score 1-8: <1% risk
#     - Score 9-14: 1-<5% risk
#     - Score 15-17: 5-<10% risk
#     - Score 18-19: 10-<20% risk
#     - Score ≥20: ≥20% risk
    
#     Parameters:
#     -----------
#     score : int
#         The calculated ALBERTA risk score
    
#     Returns:
#     --------
#     str
#         The risk category
#     float
#         The estimated risk percentage
#     """
#     if score <= 8:
#         return "<1%", 0.5
#     elif score <= 14:
#         return "1-<5%", 2.5
#     elif score <= 17:
#         return "5-<10%", 7.5
#     elif score <= 19:
#         return "10-<20%", 15
#     else:  # ≥ 20
#         return "≥20%", 25

# def calculate_alberta_scores_for_cohort(features_df, all_labs_df):
#     """
#     Calculate ALBERTA scores for an entire cohort of patients.
    
#     Parameters:
#     -----------
#     features_df : pandas.DataFrame
#         DataFrame containing patient features
#     all_labs_df : pandas.DataFrame
#         DataFrame containing lab test results
    
#     Returns:
#     --------
#     pandas.DataFrame
#         The original features DataFrame with added columns for ALBERTA score, 
#         risk category, and estimated risk percentage
#     """
#     # Create copies of the dataframes to avoid modifying originals
#     features = features_df.copy()
    
#     # Initialize new columns
#     features['alberta_score'] = None
#     features['risk_category'] = None
#     features['risk_percentage'] = None
    
#     # Calculate scores for each patient
#     for index, row in features.iterrows():
#         # Calculate the score
#         score = calculate_alberta_score(row, all_labs_df)
#         features.at[index, 'alberta_score'] = score
        
#         # Convert to risk category and percentage
#         risk_cat, risk_pct = predict_ckd_risk(score)
#         features.at[index, 'risk_category'] = risk_cat
#         features.at[index, 'risk_percentage'] = risk_pct
    
#     return features

# # Example usage:
# # result_df = calculate_alberta_scores_for_cohort(features_df, all_labs_df)
# # print(result_df[['patient_id', 'alberta_score', 'risk_category', 'risk_percentage']])